# Recipe-Aware Datasheet Generator Notebook

Flow:
1. Select machine
2. Select recipe parameters (OpcNodeIds)
3. Get last 10 runs for that machine → select one
4. Discover ALL parameters in selected run's time window (timestamp-based)
5. Get 3 most recent runs for sample collection
6. Collect up to 5,000 samples per parameter per run (~15,000 total)
7. Tag each sample with IsOk → generate datasheet with min/max/optimal

## 1. Import Libraries and Setup

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from datetime import datetime

project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

## 2. Database Connection

In [ ]:
load_dotenv()

DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT', '1433')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD', '')
DB_NAME = os.getenv('DB_NAME')

db_url = f"mssql+pyodbc://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?driver=ODBC+Driver+18+for+SQL+Server&TrustServerCertificate=yes"

engine = create_engine(db_url, echo=False)

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("Database connection successful!")
except Exception as e:
    print(f"Database connection failed: {e}")

## 3. Parameters (Papermill)

In [ ]:
machine_code = "MC_03"
recipe_params = [
    "nsu=TFE_03_MMH;s=TFE_03_MMH.10.980-Data-NDI.AN_AnnealingPower_ACT",
    "nsu=TFE_03_MMH;s=TFE_03_MMH.10.980-Data-NDI.AN_AnnealingVoltage_ACT"
]
selected_run_index = 0

print(f"Machine: {machine_code}")
print(f"Recipe params: {len(recipe_params)} selected")

## 4. Get Last 10 Runs for Machine

In [ ]:
def get_last_10_runs(machine_code):
    query = """
        SELECT TOP 10
            [RunId], [MachineCode], [StartTs], [EndTs],
            [ScopeKey] AS RecipeIdentifier
        FROM [dbo].[productionrun]
        WHERE [MachineCode] = :machine
          AND [StartTs] IS NOT NULL AND [EndTs] IS NOT NULL
        ORDER BY [StartTs] DESC
    """
    df = pd.read_sql(text(query), engine, params={"machine": machine_code})
    if not df.empty:
        df['StartTs'] = pd.to_datetime(df['StartTs'])
        df['EndTs'] = pd.to_datetime(df['EndTs'])
    return df

runs_10 = get_last_10_runs(machine_code)
print(f"Found {len(runs_10)} runs for {machine_code}")
if not runs_10.empty:
    print(runs_10[['RunId', 'StartTs', 'EndTs']].to_string())

## 5. Select Run & Discover Parameters in Time Window

In [ ]:
selected_run = runs_10.iloc[selected_run_index]
print(f"Selected run: {selected_run['RunId']}")
print(f"StartTs: {selected_run['StartTs']}")
print(f"EndTs: {selected_run['EndTs']}")

def get_all_params_in_time_window(machine_code, start_ts, end_ts):
    query = """
        SELECT DISTINCT OpcNodeId
        FROM MachineTagValue
        WHERE MachineCode = :machine
          AND SourceTimestamp BETWEEN :start_ts AND :end_ts
        ORDER BY OpcNodeId
    """
    df = pd.read_sql(text(query), engine, params={
        "machine": machine_code, "start_ts": start_ts, "end_ts": end_ts
    })
    return df['OpcNodeId'].tolist() if not df.empty else []

discovered_params = get_all_params_in_time_window(
    machine_code, selected_run['StartTs'], selected_run['EndTs']
)
print(f"\nDiscovered {len(discovered_params)} parameters in time window")
for p in discovered_params[:10]:
    print(f"  {p.replace('_ACT', '').replace('_', ' ')}")
if len(discovered_params) > 10:
    print(f"  ... and {len(discovered_params) - 10} more")

## 6. Get 3 Most Recent Runs & Collect Labeled Samples

In [ ]:
def get_last_3_runs(machine_code):
    query = """
        SELECT TOP 3
            [RunId], [MachineCode], [StartTs], [EndTs],
            [ScopeKey] AS RecipeIdentifier
        FROM [dbo].[productionrun]
        WHERE [MachineCode] = :machine
          AND [StartTs] IS NOT NULL AND [EndTs] IS NOT NULL
        ORDER BY [StartTs] DESC
    """
    df = pd.read_sql(text(query), engine, params={"machine": machine_code})
    if not df.empty:
        df['StartTs'] = pd.to_datetime(df['StartTs'])
        df['EndTs'] = pd.to_datetime(df['EndTs'])
    return df

last_3_runs = get_last_3_runs(machine_code)
print(f"Found {len(last_3_runs)} recent runs")

def get_labeled_samples(machine_code, runs, param_list, samples_per_run=5000):
    if runs.empty or not param_list:
        return pd.DataFrame()

    all_samples = []
    run_ids = runs['RunId'].tolist()
    run_ids_str = ','.join([f"'{r}'" for r in run_ids])

    quality_df = pd.read_sql(
        text(f"SELECT RunId, IsOk FROM [dbo].[ProductionRunQuality] WHERE RunId IN ({run_ids_str})"),
        engine
    )
    quality_map = dict(zip(quality_df['RunId'], quality_df['IsOk']))

    for param in param_list:
        for _, run in runs.iterrows():
            end_ts = run['EndTs'] if pd.notna(run['EndTs']) else datetime.now()
            samples_df = pd.read_sql(
                text("""
                    SELECT TOP (:limit) OpcNodeId, Value, SourceTimestamp
                    FROM MachineTagValue
                    WHERE MachineCode = :machine
                      AND OpcNodeId = :param
                      AND SourceTimestamp BETWEEN :start_ts AND :end_ts
                    ORDER BY SourceTimestamp DESC
                """),
                engine,
                params={
                    "machine": machine_code, "param": param,
                    "start_ts": run['StartTs'], "end_ts": end_ts,
                    "limit": samples_per_run
                }
            )
            if not samples_df.empty:
                samples_df['Value'] = pd.to_numeric(samples_df['Value'], errors='coerce')
                samples_df = samples_df.dropna(subset=['Value'])
                samples_df['RunId'] = run['RunId']
                samples_df['IsOk'] = quality_map.get(run['RunId'], 0)
                all_samples.append(samples_df)

    return pd.concat(all_samples, ignore_index=True) if all_samples else pd.DataFrame()

if discovered_params and not last_3_runs.empty:
    print(f"Collecting samples for {len(discovered_params)} parameters across {len(last_3_runs)} runs...")
    labeled_samples = get_labeled_samples(machine_code, last_3_runs, discovered_params, samples_per_run=5000)

    if not labeled_samples.empty:
        total = len(labeled_samples)
        ok_count = len(labeled_samples[labeled_samples['IsOk'] == 1])
        print(f"Collected {total:,} samples ({ok_count:,} OK / {total - ok_count:,} NOT OK)")
    else:
        print("No samples collected.")
else:
    labeled_samples = pd.DataFrame()

## 7. Calculate Statistics & Generate Datasheet

In [ ]:
def calculate_statistics(labeled_samples):
    if labeled_samples.empty:
        return pd.DataFrame()

    results = []
    for param in labeled_samples['OpcNodeId'].unique():
        data = labeled_samples[labeled_samples['OpcNodeId'] == param].copy()
        data['Value'] = pd.to_numeric(data['Value'], errors='coerce')
        data = data.dropna(subset=['Value'])
        if data.empty:
            continue

        all_vals = data['Value'].tolist()
        ok_vals = data[data['IsOk'] == 1]['Value'].tolist()

        results.append({
            'OpcNodeId': param,
            'ParameterName': param.replace('_recipe', '').replace('_ACT', '').replace('_', ' '),
            'MinValue': float(min(all_vals)),
            'MaxValue': float(max(all_vals)),
            'MeanValue': float(data['Value'].mean()),
            'OptimalValue': float(np.median(ok_vals)) if ok_vals else float(data['Value'].mean()),
            'StdDev': float(data['Value'].std()) if len(all_vals) > 1 else 0.0,
            'SampleCount': len(all_vals),
            'QualityOkCount': len(ok_vals),
            'QualityNotOkCount': len(all_vals) - len(ok_vals)
        })

    return pd.DataFrame(results) if results else pd.DataFrame()

if not labeled_samples.empty:
    datasheet = calculate_statistics(labeled_samples)
    if not datasheet.empty:
        print(f"Datasheet: {len(datasheet)} parameters")
        print(datasheet[['ParameterName', 'MinValue', 'OptimalValue', 'MaxValue', 'SampleCount']].head(20).to_string())
    else:
        print("No valid statistics computed.")
else:
    datasheet = pd.DataFrame()

## 8. Store Datasheet to Database

In [ ]:
if not datasheet.empty:
    recipe_key = f"custom_{'_'.join(p.split('.')[-1] for p in recipe_params[:3])}"
    try:
        with engine.begin() as conn:
            conn.execute(text("""
                DELETE FROM [model_schema].[parameter_reference_datasheet]
                WHERE MachineCode = :machine AND RecipeIdentifier = :recipe
            """), {'machine': machine_code, 'recipe': recipe_key[:255]})

            for _, row in datasheet.iterrows():
                conn.execute(text("""
                    INSERT INTO [model_schema].[parameter_reference_datasheet]
                    ([MachineCode], [RecipeIdentifier], [OpcNodeId], [ParameterName],
                     [MinValue], [OptimalValue], [MaxValue], [MeanValue], [StdDev],
                     [SampleCount], [QualityOkCount], [QualityNotOkCount])
                    VALUES (:machine, :recipe, :opc, :name,
                            :min, :opt, :max, :mean, :std, :count, :ok, :not_ok)
                """), {
                    'machine': machine_code,
                    'recipe': recipe_key[:255],
                    'opc': str(row['OpcNodeId']),
                    'name': str(row['ParameterName']),
                    'min': float(row['MinValue']),
                    'opt': float(row['OptimalValue']),
                    'max': float(row['MaxValue']),
                    'mean': float(row['MeanValue']),
                    'std': float(row['StdDev']) if pd.notna(row.get('StdDev')) else None,
                    'count': int(row['SampleCount']),
                    'ok': int(row['QualityOkCount']),
                    'not_ok': int(row['QualityNotOkCount'])
                })

        print(f"Stored {len(datasheet)} parameters for {machine_code}")
    except Exception as e:
        print(f"Error storing datasheet: {e}")

## 9. Summary Report

In [ ]:
if not datasheet.empty:
    print("=" * 80)
    print("DATASHEET SUMMARY")
    print("=" * 80)
    print(f"Machine: {machine_code}")
    print(f"Recipe Params: {len(recipe_params)} selected")
    print(f"Selected Run: {selected_run['RunId']}")
    print(f"Discovered Parameters: {len(discovered_params)}")
    print(f"Recent Runs Used: {len(last_3_runs)}")
    print(f"Total Samples: {len(labeled_samples):,}")
    print(f"Datasheet Parameters: {len(datasheet)}")

    total_ok = datasheet['QualityOkCount'].sum()
    total_not_ok = datasheet['QualityNotOkCount'].sum()
    total = total_ok + total_not_ok
    print(f"\nQuality: {int(total_ok):,} OK ({total_ok/total*100:.1f}%) / {int(total_not_ok):,} NOT OK")
else:
    print("No datasheet generated")